# Link level as stages (draft)

> A draft for review: the link level as a chain of stages, with both of today's links rebuilt from them. Nothing here is exported yet.

## The idea

Every link-level transmission goes through the same stages; only the choice at each stage differs:

| stage | role (signals in → out) | uncoded (today's `BitLevelLink`) | 5G NR (today's `SionnaLink`) | module, after review |
|---|---|---|---|---|
| source coding | payload → `bits` | `Codec` | `Codec` | `dsp.source_coding` |
| link adaptation | `sinr` → `mcs`, `num_bits_per_symbol`, `coderate`, ... | none | `NRLinkAdaptation` | `radio.link_adaptation` |
| channel coding | `bits` → `codeword` | `NoCoding` | `NRTBEncoder` (CRC + LDPC) | `dsp.channel_coding` |
| mapping | `codeword` → `x` | `Mapper(BPSK())` | `Mapper(QAM of the MCS)` | `dsp.modulation` |
| resource mapping | `h`, `power` → `h_sym` | `FlatResources` | `SubcarrierResources` | `dsp.resource_mapping` |
| channel | `x`, `h_sym`, `no` → `y` | `ApplyChannel` | `ApplyChannel` | `dsp.signal` |
| equalization | `y`, `h_sym`, `no` → `x_hat`, `no_eff` | `OneTapEqualizer` | `OneTapEqualizer` | `dsp.equalization` |
| demapping | `x_hat`, `no_eff` → `llr` | `Demapper` | `Demapper` (APP) | `dsp.modulation` |
| channel decoding | `llr` → `bits_hat` [, `crc_ok`] | `HardDecision` | `NRTBDecoder` | `dsp.channel_coding` |
| source decoding | `bits_hat` → payload | `Codec` | `Codec` | `dsp.source_coding` |

Each stage is a `Block` (see `block`) with named inputs and outputs. The stages between the bits are a `Pipeline`,
so a stage is swapped by giving another block with the same names: an LDPC code of your own, a soft BPSK detector,
an MMSE equalizer, or a Sionna block in `Named`.

**The signals.** One row per transmission of a group (`B`), as in Sionna:

* `bits` `(B, ..., k)`: the source bits; `codeword` `(B, ..., n)`: after the channel code.
* `x` `(B, ..., S)`: the symbols; `h_sym`: the channel each symbol sees, `√P` included (broadcast against `x`).
* `no` `(B,)`: each transmission's noise power, the interference added; `y`: what the receiver gets.
* `x_hat`, `no_eff`: the equalized symbols and their noise; `llr`: soft bits (Sionna's sign: positive means 1).
* `bits_hat`, and `crc_ok` when the code checks itself.
* Parameters of a group, from link adaptation: `mcs`, `num_bits_per_symbol`, `coderate`, `tb_size`,
  `num_coded_bits`, `resource_elements`.

**`Transceiver`** runs the stages for a step's transmissions. It turns payloads into bits and back, handles what
is the same for every link (nodes at the same place, outages, a signal drowned in interference), and groups the
transmissions that can go through the stages together (same size and, with link adaptation, same MCS). The two
presets, `UncodedLink` and `NRLink`, are a `Transceiver` with their stages. `LinkLevelChannel(link=...)` takes any
of them, as `SystemLevelChannel` takes any `ErrorModel`.

**What it does not cover yet.** One antenna, and a known channel. OFDM with pilots and channel estimation, and MIMO,
are more stages (`ResourceGridMapper`, `LSChannelEstimator`, `LMMSEEqualizer`, precoding), but they need `h` per
antenna and per OFDM symbol, which the fadings don't give yet: that is the next design step.

In [ ]:
import math
from collections import defaultdict
from typing import Callable, Sequence

import numpy as np

from comm_core.block import Block, Pipeline, _is_torch
from comm_core.dsp.modulation import BPSK, Modulation, SionnaModulation
from comm_core.dsp.signal import apply_channel
from comm_core.dsp.source_coding import Codec, IndexCodec
from comm_core.radio.channel import RadioChannel
from comm_core.radio.fading import Fading, effective_coefficient
from comm_core.radio.interference import SINR
from comm_core.radio.power_control import PowerControl

## The stages

### Channel coding (`dsp.channel_coding`)

`NoCoding` and `HardDecision` are the uncoded pair: the codeword is the bits, and each bit is the sign of its LLR.
The 5G NR pair wraps Sionna's `TBEncoder` and `TBDecoder` (CRC, LDPC, rate matching), built once for each transport
block size and code rate. They take those from the group's parameters, so they know nothing about MCS tables: that
is link adaptation's job.

In [ ]:
class NoCoding(Block):
    "No channel code: the codeword is the bits."

    outputs = ('codeword',)

    def call(self, bits):
        return bits


class HardDecision(Block):
    "No channel code to decode: each bit is the sign of its LLR (1 where positive)."

    outputs = ('bits_hat',)

    def call(self, llr):
        return llr > 0


class NRTBEncoder(Block):
    "5G NR transport block coding with Sionna PHY (`TBEncoder`): CRC, LDPC and rate matching, for each group's transport block."

    outputs = ('codeword',)

    def __init__(
        self,
        channel_type: str = 'PUSCH', # 'PUSCH' (uplink) or 'PDSCH' (downlink)
        num_bp_iter: int = 20, # LDPC decoder iterations (the decoder's, built with the encoder)
        device: str | None = None, # Torch device; None: Sionna's default
        precision: str = 'single', # 'single' or 'double'
    ):
        import torch
        self._torch, self.channel_type, self.num_bp_iter = torch, channel_type, num_bp_iter
        self.device, self.precision = device, precision
        self.real = torch.float64 if precision == 'double' else torch.float32
        self._coders = {}                                  # (tb_size, coded bits, rate, bits per symbol) -> (encoder, decoder)

    def coders(self, tb_size: int, num_coded_bits: int, coderate: float, num_bits_per_symbol: int) -> tuple:
        "(encoder, decoder) of a transport block, built once."
        key = (int(tb_size), int(num_coded_bits), float(coderate), int(num_bits_per_symbol))
        if key not in self._coders:
            import contextlib, io
            from sionna.phy.nr import TBDecoder, TBEncoder
            with contextlib.redirect_stdout(io.StringIO()):
                encoder = TBEncoder(target_tb_size=key[0], num_coded_bits=key[1], target_coderate=key[2],
                                    num_bits_per_symbol=key[3], channel_type=self.channel_type,
                                    precision=self.precision, device=self.device)
            decoder = TBDecoder(encoder, num_bp_iter=self.num_bp_iter, precision=self.precision, device=self.device)
            self._coders[key] = (encoder, decoder)
        return self._coders[key]

    def call(self, bits, tb_size, num_coded_bits, coderate, num_bits_per_symbol):
        encoder, _ = self.coders(tb_size, num_coded_bits, coderate, num_bits_per_symbol)
        flat = np.asarray(bits).reshape(len(bits), -1)
        padded = np.zeros((len(bits), encoder.k))          # the payload, zero-padded to the transport block
        padded[:, :flat.shape[1]] = flat
        return encoder(self._torch.as_tensor(padded, dtype=self.real, device=encoder.device))


class NRTBDecoder(Block):
    "5G NR transport block decoding with Sionna PHY (`TBDecoder`): LDPC decoding of the LLRs, then the CRC."

    outputs = ('bits_hat', 'crc_ok')

    def __init__(
        self,
        encoder: NRTBEncoder, # Its encoder (the decoder of each transport block is built with it)
    ):
        self.encoder = encoder

    def children(self) -> list:
        return []

    def call(self, llr, tb_size, num_coded_bits, coderate, num_bits_per_symbol):
        _, decoder = self.encoder.coders(tb_size, num_coded_bits, coderate, num_bits_per_symbol)
        bits, crc = decoder(llr)
        return bits.cpu().numpy().round().astype(np.int64), crc.cpu().numpy()

### Mapping (`dsp.modulation`)

`Mapper` and `Demapper` use a `Modulation`. A function of the bits per symbol instead of a modulation gives one per
MCS (the QAM of each group). The demapper gives LLRs: today's `Modulation.demodulate` gives hard bits, so a
`Modulation` gets an `llr` method (here `llr`, a function, until then).

In [ ]:
def llr(
    modulation: Modulation, # The modulation of the symbols
    x_hat, # Equalized symbols
    no_eff, # Their noise power, broadcast against them
    method: str = 'maxlog', # Sionna constellations: 'maxlog' or 'app'
):
    "Soft bits of equalized symbols (log p(1) / p(0)), (..., S * bits_per_symbol). To become `Modulation.llr`."
    if isinstance(modulation, BPSK):                     # bit 0 -> +1: log p(1)/p(0) = -4 Re(x) / no
        return -4 * np.real(x_hat) / no_eff
    if isinstance(modulation, SionnaModulation):
        demappers = modulation.__dict__.setdefault('_soft', {})
        if method not in demappers:
            from sionna.phy.mapping import Demapper as SionnaDemapper
            demappers[method] = SionnaDemapper(method, constellation=modulation.constellation, device=modulation.device,
                                               precision='double' if modulation.real is modulation._torch.float64 else 'single')
        torch = modulation._torch
        out = demappers[method](torch.as_tensor(x_hat, device=modulation.device).to(modulation.complex),
                                torch.as_tensor(no_eff, device=modulation.device).to(modulation.real))
        return out if _is_torch(x_hat) else out.cpu().numpy()
    raise NotImplementedError(f"{type(modulation).__name__} has no soft demapping")


class Mapper(Block):
    "Bits -> symbols with a `Modulation`, the last axis padded to whole symbols."

    outputs = ('x',)

    def __init__(
        self,
        modulation: Modulation | Callable[[int], Modulation], # A modulation, or one for each number of bits per symbol
    ):
        self.modulation = modulation

    def get(self, num_bits_per_symbol: int | None = None) -> Modulation:
        "The modulation (of a group's bits per symbol, if it depends on it)."
        return self.modulation if isinstance(self.modulation, Modulation) else self.modulation(int(num_bits_per_symbol))

    def call(self, codeword, num_bits_per_symbol=None):
        m = self.get(num_bits_per_symbol)
        pad = -codeword.shape[-1] % m.bits_per_symbol
        if pad:
            codeword = np.pad(codeword, [(0, 0)] * (codeword.ndim - 1) + [(0, pad)])
        return m.modulate(codeword)


class Demapper(Block):
    "Equalized symbols -> LLRs, with the transmitter's modulation."

    outputs = ('llr',)

    def __init__(
        self,
        mapper: Mapper, # The transmitter's mapper (its modulation)
        method: str = 'maxlog', # Sionna constellations: 'maxlog' or 'app'
    ):
        self.mapper, self.method = mapper, method

    def children(self) -> list:
        return []

    def call(self, x_hat, no_eff, num_bits_per_symbol=None):
        return llr(self.mapper.get(num_bits_per_symbol), x_hat, no_eff, self.method)

### Resource mapping (`dsp.resource_mapping`)

Which channel each symbol sees, from the fading's `h` at each subcarrier, `(B, K)`, with the transmit power:

* `FlatResources`: one carrier. Every symbol sees one coefficient, `h` if flat, else `√(mean |h|²)`.
* `SubcarrierResources`: OFDM without its waveform. The symbols fill the subcarriers in turn, each with its
  subcarrier's `h`; the cyclic prefix is taken to be ideal.

In [ ]:
def _like(values, x):
    "values (numpy) as x's kind: a tensor of x's complex type on its device, or numpy."
    if _is_torch(x):
        import torch
        return torch.as_tensor(values, dtype=x.dtype, device=x.device)
    return values


class FlatResources(Block):
    "One carrier: every symbol of a transmission sees one coefficient (√(mean |h|²) over the subcarriers), √P included."

    outputs = ('h_sym',)

    def call(self, x, h, power):
        c = np.array([effective_coefficient(h[b]) * float(power[b]) ** 0.5 for b in range(len(h))])
        return _like(c.reshape((len(c),) + (1,) * (x.ndim - 1)), x)


class SubcarrierResources(Block):
    "OFDM without its waveform: the symbols fill the subcarriers in turn, each seeing its subcarrier's h; √P included."

    outputs = ('h_sym',)

    def call(self, x, h, power):
        n = x.shape[-1]
        return _like(np.stack([np.resize(h[b], n) * math.sqrt(float(power[b])) for b in range(len(h))]), x)

### The channel and equalization (`dsp.signal`, `dsp.equalization`)

`ApplyChannel` is `y = h x + n`, with each transmission's noise `no` (the interference included, as Gaussian noise).
`OneTapEqualizer` divides each symbol by its channel, knowing it (perfect channel state information).

In [ ]:
def _noise(no, x):
    "Each transmission's noise power, broadcast against its symbols."
    if _is_torch(x):
        import torch
        real = x.real.dtype
        return torch.as_tensor(np.asarray(no, float), dtype=real, device=x.device).reshape((len(no),) + (1,) * (x.dim() - 1))
    return np.asarray(no, float).reshape((len(no),) + (1,) * (x.ndim - 1))


class ApplyChannel(Block):
    "What the receiver gets: y = h x + n, n ~ CN(0, no) with each transmission's noise power (the interference included)."

    outputs = ('y',)

    def call(self, x, h_sym, no):
        if _is_torch(x):
            import torch
            gen = torch.Generator(device=x.device).manual_seed(int(self.rng.integers(2 ** 62)))
            real = x.real.dtype
            noise = torch.complex(torch.randn(x.shape, generator=gen, device=x.device, dtype=real),
                                  torch.randn(x.shape, generator=gen, device=x.device, dtype=real))
            return h_sym * x + noise * torch.sqrt(_noise(no, x) / 2)
        return np.stack([apply_channel(x[b], h_sym[b], 1.0, float(no[b]), self.rng) for b in range(len(x))])


class OneTapEqualizer(Block):
    "Each symbol divided by its channel, which the receiver knows: x̂ = y / h, with noise no / |h|²."

    outputs = ('x_hat', 'no_eff')

    def call(self, y, h_sym, no):
        if _is_torch(y):
            return y / h_sym, _noise(no, y) / (h_sym.abs() ** 2).clamp_min(1e-30)
        return y / h_sym, _noise(no, y) / np.maximum(np.abs(h_sym) ** 2, 1e-30)

### Link adaptation (`radio.link_adaptation`)

`NRLinkAdaptation` gives each transmission its MCS and what follows from it: the QAM order, the code rate and the 5G
NR allocation. The MCS is fixed, or the highest that Sionna SYS (`SionnaErrorModel`) predicts to meet the BLER
target at the transmission's SINR (one per broadcast). The stages after it read the parameters, so the same coding
and mapping stages work with another table or another way of choosing.

In [ ]:
class NRLinkAdaptation(Block):
    "Each transmission's MCS (fixed, or meeting a BLER target at its SINR) and its 5G NR allocation."

    outputs = ('mcs', 'num_bits_per_symbol', 'coderate', 'resource_elements', 'tb_size', 'num_coded_bits')

    def __init__(
        self,
        error_model, # A `SionnaErrorModel`: the MCS table, the candidates, the BLER target or a fixed MCS
    ):
        self.error_model = error_model

    def call(self, sinr, num_bits, group=None):
        _, _, rated = self.error_model(list(sinr), np.asarray(num_bits), group=group)
        out = {name: [] for name in self.outputs}
        for i, b in zip(rated, num_bits):
            m = i['mcs']
            q, r = self.error_model.table.mcs(m)
            num_re, tb_size = self.error_model.allocation(int(b), m)
            for name, v in zip(self.outputs, (m, q, r, num_re, tb_size, num_re * q)):
                out[name].append(v)
        return {name: np.array(v) for name, v in out.items()}

## `Transceiver`

The stages for a step's transmissions. For each transmission:

1. The payload becomes bits (`codec`).
2. **Shortcuts, the same for every link:** an outage is not sent; nodes at the same place have a perfect link; a
   signal with no power, or drowned by an interferer at the receiver's place, is lost.
3. **Link adaptation** (`adaptation`, optional) gives each transmission its parameters, from its SINR.
4. **Groups.** The transmissions with the same bits' shape and parameters go through the stages together
   (`batch=True`); with `batch=False`, one at a time.
5. The bits that come out become the payload. Delivery is the CRC's (`crc_ok`) when the code has one; without,
   payloads arrive with their bit errors, or are dropped with `drop_corrupted` (an ideal CRC).

The stages share the transceiver's random generator, so its seed decides the noise. `report` names the group
parameters added to each result's `info`.

In [ ]:
def _rows(x, rows):
    return x[rows] if isinstance(x, np.ndarray) else [x[k] for k in rows]


def _size(x) -> int:
    return int(x.numel()) if _is_torch(x) else int(np.size(x))


def _truncate(bits_hat, bits):
    "The decoded bits of one transmission, without the padding."
    bits_hat = np.asarray(bits_hat)
    if bits_hat.ndim == bits.ndim:                      # padded along the last axis (to whole symbols)
        return bits_hat[..., :bits.shape[-1]]
    return bits_hat.reshape(-1)[:bits.size].reshape(bits.shape)   # flattened into a transport block


class Transceiver(Block):
    "Link level: payload -> bits -> the stages (coding, mapping, the channel, equalization, demapping, decoding) -> payload."

    outputs = ('success', 'payload', 'error_rate', 'num_bits', 'num_symbols', 'info')

    def __init__(
        self,
        stages: Sequence[Block], # From `bits` to `bits_hat` (and `crc_ok` if the code checks itself)
        codec: Codec | None = None, # Payload <-> bits; None: `IndexCodec(256)`
        adaptation: Block | None = None, # Each transmission's parameters (e.g. its MCS), from `sinr` and `num_bits`
        batch: bool = True, # Transmissions with the same shape and parameters through the stages together
        drop_corrupted: bool = False, # Without a CRC: lose the packets with bit errors instead of delivering them
        report: Sequence[str] = (), # Parameters added to each result's info
    ):
        self.codec = codec if codec is not None else IndexCodec()
        self.stages = stages if isinstance(stages, Pipeline) else Pipeline(list(stages))
        self.adaptation, self.batch, self.drop_corrupted, self.report = adaptation, batch, drop_corrupted, tuple(report)
        self._share()

    def _share(self) -> None:
        for block in self.stages.walk():
            block.rng = self.rng

    def reset(self, seed: int | None = None) -> None:
        super().reset(seed)
        self._share()

    @property
    def needs_sinr(self) -> bool:
        "Whether the channel must give the SINR (for link adaptation)."
        return self.adaptation is not None and 'sinr' in self.adaptation.inputs

    @property
    def bits_per_symbol(self) -> int:
        "Of a fixed modulation (counts the symbols of what is not sent through the stages); 1 if it varies."
        for block in self.stages:
            if isinstance(block, Mapper) and isinstance(block.modulation, Modulation):
                return block.modulation.bits_per_symbol
        return 1

    def call(
        self,
        transmission, # The transmissions (their payloads)
        h: np.ndarray, # h of each at each subcarrier, (n, K)
        power: np.ndarray, # Their transmit powers
        noise_power: float, # Noise power at the receivers (per symbol)
        interference: np.ndarray | None = None, # Power of the other transmissions at each receiver (added to the noise)
        active: np.ndarray | None = None, # Whether each one is sent (None: all); the others only count their bits
        group: Sequence | None = None, # The broadcast of each, None for a unicast (link adaptation: one MCS per broadcast)
        sinr: np.ndarray | None = None, # Their SINR at each subcarrier, for link adaptation
    ) -> tuple:
        "(success, payload, error_rate, num_bits, num_symbols, info) of each transmission."
        n = len(transmission)
        success, payload, error = np.zeros(n, dtype=bool), [None] * n, np.ones(n)
        num_bits, num_symbols, info = np.zeros(n, np.int64), np.zeros(n, np.int64), [{} for _ in range(n)]
        interference = np.zeros(n) if interference is None else np.asarray(interference, float)
        bits, todo = {}, []
        for k, t in enumerate(transmission):
            bits[k] = np.asarray(self.codec.encode(t.message.payload))
            num_bits[k], num_symbols[k] = bits[k].size, -(-int(bits[k].size) // self.bits_per_symbol)
            if active is not None and not active[k]:
                continue
            if np.isinf(h[k]).any():                     # same place: a perfect link
                success[k], payload[k], error[k], num_symbols[k], info[k] = True, t.message.payload, 0.0, 0, {'bit_errors': 0}
                continue
            noise = noise_power + float(interference[k])
            if not (float(power[k]) * abs(effective_coefficient(h[k])) ** 2 > 0 and math.isfinite(noise)):
                continue                                 # nothing to detect: no signal, or drowned (an interferer at the receiver's place)
            todo.append(k)
        if not todo:
            return success, payload, error, num_bits, num_symbols, info
        params = {}
        if self.adaptation is not None:
            params = self.adaptation(sinr=_rows(sinr, todo), num_bits=num_bits[todo],
                                     group=None if group is None else _rows(group, todo))
        groups = defaultdict(list)
        for i, k in enumerate(todo):
            key = (bits[k].shape, *(params[p][i].item() for p in params)) if self.batch else (k,)
            groups[key].append(i)
        for rows in groups.values():
            ks = [todo[i] for i in rows]
            state = dict(bits=np.stack([bits[k] for k in ks]), h=h[ks], power=np.asarray(power)[ks],
                         no=noise_power + interference[ks], **{p: v[rows[0]] for p, v in params.items()})
            out = self.stages(**{name: state[name] for name in self.stages.inputs if name in state})
            crc = out.get('crc_ok')
            for row, k in enumerate(ks):
                decoded = _truncate(out['bits_hat'][row], bits[k])
                errors = int((decoded != bits[k]).sum())
                success[k] = bool(crc[row]) if crc is not None else not (self.drop_corrupted and errors)
                payload[k] = self.codec.decode(decoded) if success[k] else None
                error[k], num_symbols[k] = errors / max(int(bits[k].size), 1), _size(out['x'][row])
                info[k] = {**{p: params[p][rows[row]].item() for p in self.report}, 'bit_errors': errors}
        return success, payload, error, num_bits, num_symbols, info

## The presets (`radio.link`)

Today's two links, as a `Transceiver` with their stages:

In [ ]:
class UncodedLink(Transceiver):
    "Uncoded bits on one carrier: payload -> bits -> symbols -> h √P x + n -> detected bits -> payload (bit errors pass)."

    def __init__(
        self,
        codec: Codec | None = None, # Payload <-> bits; None: `IndexCodec(256)`
        modulation: Modulation | None = None, # Bits <-> symbols; None: `BPSK()`
        drop_corrupted: bool = False, # Lose the packets with bit errors (an ideal CRC) instead of delivering them
    ):
        self.modulation = modulation if modulation is not None else BPSK()
        mapper = Mapper(self.modulation)
        super().__init__([NoCoding(), mapper, FlatResources(), ApplyChannel(), OneTapEqualizer(), Demapper(mapper),
                          HardDecision()], codec, batch=False, drop_corrupted=drop_corrupted)


MIN_LDPC_CODERATE = 0.2                                  # Sionna's 5G LDPC encoder: rates below 1/5 are not supported


class NRLink(Transceiver):
    "5G NR with Sionna PHY: an MCS per transmission, CRC + LDPC, QAM, h at each subcarrier, one-tap equalization, APP demapping, LDPC decoding."

    def __init__(
        self,
        codec: Codec | None = None, # Payload <-> bits; None: `IndexCodec(256)`
        mcs_index: int | None = None, # A fixed MCS; None: link adaptation
        bler_target: float = 0.1, # Link adaptation's target error rate
        mcs_table_index: int = 1, # 5G NR MCS table: 1 (up to 64-QAM) or 2 (up to 256-QAM)
        mcs_category: int = 0, # 0: PUSCH (uplink), 1: PDSCH (downlink)
        num_bp_iter: int = 20, # LDPC decoder iterations
        device: str | None = None, # Torch device; None: Sionna's default (a GPU if there is one)
        precision: str = 'single', # 'single' or 'double'
    ):
        from comm_core.radio.link_adaptation import NRMCSTable
        from comm_core.radio.system.phy_abstraction import SionnaErrorModel
        supported = NRMCSTable(mcs_table_index, mcs_category, min_coderate=MIN_LDPC_CODERATE, device=device).indices
        if mcs_index is not None and mcs_index not in supported:
            raise ValueError(f"MCS {mcs_index}: not one Sionna PHY can encode (code rates below 1/5 are not supported); "
                             f"one of {supported}")
        self.error_model = SionnaErrorModel(mcs_index=mcs_index, bler_target=bler_target, mcs_table_index=mcs_table_index,
                                            mcs_category=mcs_category, device=device, precision=precision,
                                            mcs_indices=supported)
        qam = {}
        def modulation(q):                                 # the QAM of an MCS, built once
            if q not in qam:
                qam[q] = SionnaModulation('qam', q, device=self.error_model.device, precision=precision)
            return qam[q]
        mapper = Mapper(modulation)
        encoder = NRTBEncoder('PUSCH' if mcs_category == 0 else 'PDSCH', num_bp_iter, self.error_model.device, precision)
        super().__init__([encoder, mapper, SubcarrierResources(), ApplyChannel(), OneTapEqualizer(),
                          Demapper(mapper, 'app'), NRTBDecoder(encoder)], codec,
                         adaptation=NRLinkAdaptation(self.error_model), report=('mcs', 'resource_elements', 'tb_size'))

## `LinkLevelChannel(link=...)`

One channel for every link, as `SystemLevelChannel` is one channel for every `ErrorModel`. It adds the `SINR` block
when the link needs it (link adaptation). It replaces today's `LinkLevelChannel` (uncoded only) and
`SionnaLinkChannel`.

In [ ]:
class LinkLevelChannel(RadioChannel):
    "Every bit simulated, through a link's stages (`UncodedLink`, `NRLink`, or a `Transceiver` of your own)."

    def __init__(
        self,
        link: Transceiver | None = None, # The link; None: `UncodedLink()`
        fading: Fading | None = None, # The coefficient h of a transmission; None: h = 1
        power_control: PowerControl | None = None, # Transmit power and outages; None: the link's tx_power_w
        noise_power: float = 1.0, # Noise power at the receiver, per symbol
        seed: int | None = None, # Seed of the noise and the fading
        interference: bool = False, # A step's transmissions share the medium (see `RadioChannel`)
        half_duplex: bool = True, # With interference: a node that transmits in a step cannot receive in it
    ):
        self.link = link if link is not None else UncodedLink()
        self.sinr = SINR()
        super().__init__(fading, power_control, noise_power, seed, interference, half_duplex)

    def receiver(self) -> list:
        "The link, after the SINR if it needs it."
        return [self.sinr, self.link] if self.link.needs_sinr else [self.link]

## Checks: the same results as today's links

With the same seeds, the stages give exactly what `BitLevelLink` and `SionnaLink` give: the same noise, the same
decisions, the same metadata.

In [ ]:
from comm_core.core import Message, Transmission
from comm_core.net.topology import Link
from comm_core.radio.fading import NoFading, RayleighFading
from comm_core.radio.link.channel import LinkLevelChannel as OldLinkLevelChannel, SionnaLinkChannel
from comm_core.radio.power_control import ChannelInversion
from comm_core.radio.ray_tracing import RayTracedFading

In [ ]:
def step(senders, payloads):
    return [(Transmission(Message(s, r, p), s, r, 0.0), Link(s, r)) for (s, r), p in zip(senders, payloads)]

def same(a, b):
    "Two lists of results, field by field."
    for x, y in zip(a, b, strict=True):
        assert (x.success, x.size_bits, x.error_rate, x.latency, x.energy, x.snr_db) == (y.success, y.size_bits, y.error_rate, y.latency, y.energy, y.snr_db)
        assert x.metadata == y.metadata
        assert (x.payload is None and y.payload is None) or np.array_equal(x.payload, y.payload)

rng = np.random.default_rng(0)
pairs = [('a', 'b'), ('b', 'c'), ('c', 'a'), ('a', 'c')] * 5
payloads = [rng.integers(0, 256, size=rng.integers(1, 40)) for _ in pairs]
for fading, power_control in ((NoFading(0.3), None), (RayleighFading(seed=1), None), (RayleighFading(seed=2), ChannelInversion(3, max_power=4))):
    for drop in (False, True):
        old = OldLinkLevelChannel(fading=fading, power_control=power_control, noise_power=0.5, drop_corrupted=drop)
        new = LinkLevelChannel(UncodedLink(drop_corrupted=drop), fading=fading, power_control=power_control, noise_power=0.5)
        for seed in (0, 1):
            old.reset(seed); a = old.transmit_batch(step(pairs, payloads))
            new.reset(seed); b = new.transmit_batch(step(pairs, payloads))
            same(a, b)
assert 0 < np.mean([r.error_rate for r in a if r.error_rate is not None]) < 0.5 and any(r.metadata.get('outage') for r in a)

In [ ]:
# with interference, half-duplex and nodes at the same place
xs = np.arange(5)
d = np.abs(np.subtract.outer(xs, xs)).astype(float)
table = RayTracedFading(np.where(d > 0, 1e-3 / np.maximum(d, 1) ** 1.5, 0), [(x, 0) for x in xs],
                        node_positions={'a': (0, 0), 'b': (1, 0), 'c': (2, 0), 'd': (3, 0), 'e': (3, 0)})
pairs = [('a', 'b'), ('c', 'd'), ('d', 'e'), ('b', 'a'), ('e', 'c')]
payloads = [np.arange(64) % 256] * len(pairs)
old = OldLinkLevelChannel(fading=table, noise_power=1e-10, interference=True, seed=0)
new = LinkLevelChannel(fading=table, noise_power=1e-10, interference=True, seed=0)
same(old.transmit_batch(step(pairs, payloads)), new.transmit_batch(step(pairs, payloads)))

In [ ]:
#| sionna
# any Sionna constellation, uncoded: 16-QAM
pairs = [('a', 'b'), ('b', 'c'), ('c', 'a')] * 4
payloads = [rng.integers(0, 256, size=rng.integers(1, 40)) for _ in pairs]
for seed in (0, 1):
    qam = SionnaModulation('qam', 4, device='cpu')
    old = OldLinkLevelChannel(modulation=qam, fading=RayleighFading(seed=3), noise_power=0.05, seed=seed)
    new = LinkLevelChannel(UncodedLink(modulation=qam), fading=RayleighFading(seed=3), noise_power=0.05, seed=seed)
    a, b = old.transmit_batch(step(pairs, payloads)), new.transmit_batch(step(pairs, payloads))
    same(a, b)
assert 0 < np.mean([r.error_rate for r in a]) < 0.5

In [ ]:
#| sionna
# 5G NR: link adaptation and a fixed MCS, frequency-selective fading (two subcarriers), sizes that share and don't share a group
class TwoSubcarriers(Fading):
    def __init__(self, rng):
        self.rng_ = rng
    def coefficient(self, transmission, link):
        return 1.0
    def response(self, transmission, link):
        return (self.rng_.standard_normal(2) + 1j * self.rng_.standard_normal(2)) * 0.1

pairs = [('a', 'b'), ('b', 'c'), ('c', 'a'), ('a', 'c'), ('b', 'a')] * 2
payloads = [np.arange(n) % 256 for n in (100, 100, 40, 100, 40, 250, 100, 40, 100, 100)]
for mcs in (None, 9):
    for seed in (0, 1):
        old = SionnaLinkChannel(fading=TwoSubcarriers(np.random.default_rng(seed)), noise_power=10 ** -1.8, mcs_index=mcs, device='cpu', seed=seed)
        new = LinkLevelChannel(NRLink(mcs_index=mcs, device='cpu'), fading=TwoSubcarriers(np.random.default_rng(seed)), noise_power=10 ** -1.8, seed=seed)
        a, b = old.transmit_batch(step(pairs, payloads)), new.transmit_batch(step(pairs, payloads))
        same(a, b)
    print(f"MCS {mcs}: delivered {sum(r.success for r in a)} of {len(a)}, MCSs {sorted({r.metadata['mcs'] for r in a})}")
assert [type(x).__name__ for x in new.pipeline()] == ['TwoSubcarriers', 'FixedPower', 'SINR', 'NRLink', 'Airtime']

In [ ]:
#| sionna
# a broadcast: one MCS for its copies; interference
nodes = ['a', 'b', 'c', 'd']
pairs = [('a', 'b'), ('a', 'c'), ('a', 'd'), ('c', 'd')]
payloads = [np.arange(100) % 256] * 4
def broadcast_step():
    return [(Transmission(Message(s, r, p, metadata={'broadcast': 7} if s == 'a' else {}), s, r, 0.0), Link(s, r))
            for (s, r), p in zip(pairs, payloads)]
old = SionnaLinkChannel(fading=table, noise_power=1e-9, interference=True, device='cpu', seed=0)
new = LinkLevelChannel(NRLink(device='cpu'), fading=table, noise_power=1e-9, interference=True, seed=0)
table.set_positions({'a': (0, 0), 'b': (1, 0), 'c': (3, 0), 'd': (4, 0)})
a, b = old.transmit_batch(broadcast_step()), new.transmit_batch(broadcast_step())
same(a, b)
assert a[1].metadata['half_duplex'] and len({r.metadata['mcs'] for r in (a[0], a[2])}) == 1   # c transmits: deaf; one MCS for the others